# GPU preprocessing alternatives: Colab validation

Partial GPU validation for `specs/2026-10-01-gpu-preprocess-alternatives` on a Colab GPU runtime
(no Docker). It covers only the checks that need a GPU but not Triton:

* **T-3** DALI letterbox conventions (pad rounding, resampling, degenerate input) and the DALI reference tensors
* **T-2** TensorRT acceptance of the in-graph letterbox (assumption A-1), against the reference and DALI
* **V-5** the shared CUDA kernel (`cuda_preprocess_dump`) against the DALI tensors
* **V-14 (kernel)** `compute-sanitizer` memcheck + racecheck on the kernel

Backend loading, V2 CUDA IPC, detection parity, soak and benchmarks still need a Docker GPU host.
The toolchain follows the NGC 26.08 pins: CUDA 13.3 toolkit (TensorRT's build series), TensorRT
11.2.1.2 and DALI 2.2.0 from pip, with `cuda-compat` when the driver predates R580. The summary
records every version actually used.

**Setup:** Runtime → Change runtime type → GPU (High-RAM if offered). Upload `tritonic.zip` to
`/content`, then Runtime → Run all. The last cell prints `SUMMARY_JSON_BEGIN … END`.

In [ ]:
# 1. GPU and driver.
import json, os, pathlib, re, shutil, subprocess
# NGC 26.08 pins (same as olibartfast/rf-detr-cpp-inference versions.env): TensorRT 11.2.1.2
# built against CUDA 13.3, DALI 2.2.0 (the DALI inside tritonserver:26.08).
CUDA_VERSION, CUDA_DASHED, CUDA_MIN_DRIVER = "13.3", "13-3", 580
TENSORRT_VERSION, DALI_VERSION = "11.2.1.2", "2.2.0"
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
gpu = subprocess.check_output(["nvidia-smi", "--query-gpu=name,driver_version,compute_cap",
                               "--format=csv,noheader"], text=True).splitlines()[0].strip()
driver_major = int(gpu.split(",")[1].strip().split(".")[0])
SM = gpu.split(",")[2].strip().replace(".", "")
NEED_COMPAT = driver_major < CUDA_MIN_DRIVER
print({"gpu": gpu, "sm": SM, "driver_major": driver_major, "cuda_compat": NEED_COMPAT})

In [ ]:
# 2. Unpack the repository uploaded as /content/tritonic.zip.
%cd /content
!rm -rf /content/tritonic && mkdir -p /content/tritonic /content/results && unzip -q /content/tritonic.zip -d /content/tritonic
%cd /content/tritonic
!cat COMMIT

In [ ]:
# 3. Toolchain: CUDA 13.3 toolkit (nvcc, compute-sanitizer) + compat if the driver is older,
#    OpenCV C++ for the kernel tool. ~5 min.
pkgs = f"cuda-toolkit-{CUDA_DASHED} libopencv-dev" + (f" cuda-compat-{CUDA_DASHED}" if NEED_COMPAT else "")
!wget -q -O /tmp/cuda-keyring.deb https://developer.download.nvidia.com/compute/cuda/repos/ubuntu$(. /etc/os-release; echo ${VERSION_ID//./})/x86_64/cuda-keyring_1.1-1_all.deb && dpkg -i /tmp/cuda-keyring.deb > /dev/null
!apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq --no-install-recommends {pkgs} > /tmp/apt.log 2>&1; tail -3 /tmp/apt.log
CUDA_HOME = f"/usr/local/cuda-{CUDA_VERSION}"
compat = f"{CUDA_HOME}/compat:" if NEED_COMPAT else ""
os.environ["PATH"] = f"{CUDA_HOME}/bin:" + os.environ["PATH"]
os.environ["LD_LIBRARY_PATH"] = f"{compat}{CUDA_HOME}/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")
!nvcc --version | tail -2
# Probe: the driver (with compat if needed) must run CUDA 13 code before anything else is tried.
pathlib.Path("/tmp/probe.cu").write_text("#include <cstdio>\n#include <cuda_runtime.h>\nint main(){int d=0,r=0;cudaDriverGetVersion(&d);cudaRuntimeGetVersion(&r);int n=0;cudaError_t e=cudaGetDeviceCount(&n);printf(\"driver=%d runtime=%d devices=%d %s\\n\",d,r,n,cudaGetErrorString(e));return e!=cudaSuccess;}\n")
!nvcc -arch=sm_{SM} /tmp/probe.cu -o /tmp/probe && /tmp/probe

In [ ]:
# 4. Python packages pinned to NGC 26.08: DALI, TensorRT, plus CuPy for TensorRT buffers.
!pip install -q --extra-index-url https://pypi.nvidia.com "nvidia-dali-cuda130=={DALI_VERSION}" "tensorrt-cu13=={TENSORRT_VERSION}" cupy-cuda13x onnx opencv-python-headless numpy 2>&1 | tail -3
import importlib.metadata as md
versions = {p: md.version(p) for p in ("nvidia-dali-cuda130", "tensorrt-cu13", "cupy-cuda13x", "onnx")}
versions["nvcc"] = subprocess.check_output(["nvcc", "--version"], text=True).splitlines()[-1]
print(versions)

In [ ]:
# 5. Spike T-3: DALI letterbox conventions; writes the DALI reference tensors.
!python3 integration-tests/gpu-preprocess/dali_letterbox_spike.py --tensor-dir /content/results/dali --output /content/results/dali_letterbox.json
t3 = json.load(open("/content/results/dali_letterbox.json"))
PAD = t3["pad_rounding"] if t3["pad_rounding"] in ("floor", "ceil") else "floor"
print("pad rounding:", t3["pad_rounding"])
print(json.dumps(t3["geometry"])[:3000])
print(json.dumps(t3["resampling"])[:3000])
print(json.dumps(t3["degenerate"]))

In [ ]:
# 6. Spike T-2: TensorRT builds and reproduces the letterbox graph (Python API path).
!python3 integration-tests/gpu-preprocess/trt_letterbox_spike.py --pad-rounding {PAD} --dali-dir /content/results/dali --output /content/results/trt_letterbox.json
t2 = json.load(open("/content/results/trt_letterbox.json"))
print({k: t2.get(k) for k in ("runner", "build_ok", "passed")})
for case in t2.get("cases", []):
    print(case["case"], {k: (case[k]["passed"], round(case[k]["max_abs"], 5), round(case[k]["mean_abs"], 6))
                         for k in ("vs_reference", "vs_dali") if k in case})
print(t2.get("build_log_tail", "")[-2000:])

In [ ]:
# 7. V-5: build the shared kernel tool with nvcc 13.3, then compare its tensors with DALI's.
!nvcc -std=c++17 -O2 -lineinfo -arch=sm_{SM} -Iinclude src/cuda/letterbox.cu tests/tools/cuda_preprocess_dump.cpp $(pkg-config --cflags --libs opencv4) -o /content/cuda_preprocess_dump && echo BUILD_OK
!/content/cuda_preprocess_dump --pad-rounding={PAD} /content/results/kernel data/images/bus.jpg data/images/horses.jpg data/images/person.jpg data/images/mug.jpg
!python3 integration-tests/gpu-preprocess/compare_tensors.py --reference /content/results/dali --actual /content/results/kernel --output /content/results/tensors_kernel.json

In [ ]:
# 8. V-14 (kernel part): compute-sanitizer memcheck and racecheck.
sanitizer = {}
for tool in ("memcheck", "racecheck"):
    r = subprocess.run(["compute-sanitizer", "--tool", tool, "--error-exitcode", "1", "/content/cuda_preprocess_dump",
                        f"--pad-rounding={PAD}", f"/content/results/kernel_{tool}",
                        "data/images/bus.jpg", "data/images/mug.jpg"], capture_output=True, text=True)
    lines = (r.stdout + r.stderr).splitlines()
    sanitizer[tool] = {"exit": r.returncode, "summary": [l for l in lines if "SUMMARY" in l]}
    pathlib.Path(f"/content/results/sanitizer_{tool}.log").write_text(r.stdout + r.stderr)
print(sanitizer)

In [ ]:
# 9. Summary (copied into the spec's validation evidence).
tk = json.load(open("/content/results/tensors_kernel.json"))
summary = {
    "gpu": gpu, "cuda_compat": NEED_COMPAT, "versions": versions,
    "T3_pad_rounding": t3["pad_rounding"],
    "T3_geometry_matches": {r: all(g[r]["matches"] for g in t3["geometry"]) for r in ("floor", "ceil")},
    "T3_resampling": {e["case"]: (round(e[PAD]["max_abs"], 5), round(e[PAD]["mean_abs"], 6), round(e[PAD]["p99_abs"], 5))
                      for e in t3["resampling"]},
    "T3_degenerate": [(d["input_hw"], d["outcome"], d.get("message", "")[:120]) for d in t3["degenerate"]],
    "T2_runner": t2.get("runner"), "T2_build_ok": t2["build_ok"], "T2_passed": t2["passed"],
    "T2_cases": {c["case"]: {k: (c[k]["passed"], round(c[k]["max_abs"], 5), round(c[k]["mean_abs"], 6))
                             for k in ("vs_reference", "vs_dali") if k in c} for c in t2.get("cases", [])},
    "V5_kernel_vs_dali": {r["tensor"]: (r["passed"], round(r.get("max_abs", -1), 5), round(r.get("mean_abs", -1), 6),
                                        round(r.get("p99_abs", -1), 5)) for r in tk["results"]},
    "V14_kernel": {k: (v["exit"], v["summary"]) for k, v in sanitizer.items()},
}
json.dump(summary, open("/content/results/summary.json", "w"), indent=2)
print("SUMMARY_JSON_BEGIN"); print(json.dumps(summary)); print("SUMMARY_JSON_END")